# MTG lncRNA Pipeline

### Setting up environment

In [1]:
# libraries
import sys
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import plotly.io as pio
pio.renderers.default = "notebook"
import nsforest as ns
from nsforest import utils
import celltypist as ct
import scipy.sparse as sp

/opt/miniconda3/envs/nsforest/lib/python3.11/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [3]:
# configs
code_folder = "/Users/vbecker/Desktop/NSForest-ncRNA"
sys.path.insert(0, os.path.abspath(code_folder))

data_folder = "../beckersv_data/nsf_mtg/"
file = data_folder + "nsf_mtg.h5ad"

output_folder = "../beckersv_output_mtg/"

annotate_file = "../gencode_annotation/gencode_annotation/gencode.v47.annotation_txLength.csv"

to_downsample = True
to_downsample_num = 100000

seed = 1990

### Data Exploration

#### Loading AnnData file

**Note:** The MTG dataset is deconstructed, so instead of an easy import, we're building it instead.

In [4]:
# read in counts
exon = pd.read_csv(data_folder + "human_MTG_2018-06-14_exon-matrix.csv", index_col=0)
intron = pd.read_csv(data_folder + "human_MTG_2018-06-14_intron-matrix.csv", index_col=0)
assert exon.index.equals(intron.index)
assert exon.columns.equals(intron.columns)

In [5]:
# convert to cell x gene format
gene_ids = exon.index.astype(str)
cell_ids = exon.columns.astype(str)

exon = exon.T
intron = intron.T

# combine
X = exon + intron

In [9]:
gene_ids

Index(['353007', '353008', '353009', '353010', '389180', '1', '503538',
       '29974', '2', '144571',
       ...
       '158586', '79364', '440590', '100131879', '79699', '7791', '23140',
       '26009', '259265', '150478'],
      dtype='object', length=50281)

In [6]:
# Convert to sparse matrices
X_sparse = sp.csr_matrix(X.values)
exon_sparse = sp.csr_matrix(exon.values)
intron_sparse = sp.csr_matrix(intron.values)

In [7]:
# load sample metadata
samples = pd.read_csv(data_folder + "human_MTG_2018-06-14_samples-columns.csv")

samples["sample_name"] = samples["sample_name"].astype(str)

obs = (
    samples
    .set_index("sample_name")
    .loc[cell_ids]
    .copy()
)

In [12]:
# load gene metadata
genes = pd.read_csv(data_folder + "human_MTG_2018-06-14_genes-rows.csv")

genes["entrez_id"] = genes["entrez_id"].astype(str)

var = (
    genes
    .set_index("entrez_id")
    .loc[gene_ids]
    .copy()
)

In [14]:
# create adata
adata_raw = ad.AnnData(
    X = X_sparse,
    obs = obs,
    var = var,
    dtype = np.int32,
)
adata_raw

AnnData object with n_obs × n_vars = 15928 × 50281
    obs: 'sample_id', 'sample_type', 'organism', 'donor', 'sex', 'age_days', 'brain_hemisphere', 'brain_region', 'brain_subregion', 'facs_date', 'facs_container', 'facs_sort_criteria', 'rna_amplification_set', 'library_prep_set', 'library_prep_avg_size_bp', 'seq_name', 'seq_tube', 'seq_batch', 'total_reads', 'percent_exon_reads', 'percent_intron_reads', 'percent_intergenic_reads', 'percent_rrna_reads', 'percent_mt_exon_reads', 'percent_reads_unique', 'percent_synth_reads', 'percent_ecoli_reads', 'percent_aligned_reads_total', 'complexity_cg', 'genes_detected_cpm_criterion', 'genes_detected_fpkm_criterion', 'class', 'cluster'
    var: 'gene', 'chromosome', 'gene_name', 'mouse_homologenes'

#### Looking at sample labels

In [ ]:
adata_raw.obs_names # sample names

In [ ]:
adata_raw.obs.columns # sample metadata

#### Looking at genes

**Note:** `adata.var_names` must be unique. If there is a problem, usually it can be solved by assigning `adata.var.index = adata.var["ensembl_id"]`. 

In [ ]:
adata_raw.var_names # gene names

In [ ]:
adata_raw.var.columns # gene metadata
adata_raw.var["feature_type"].value_counts().head(5)

### Downsampling and Subsetting

#### Defining `cluster_header` as cell type annotation and gene comparison for subset.

**Note:** Some datasets have multiple annotations per sample (ex. "broad_cell_type" and "granular_cell_type"). NS-Forest can be run on multiple `cluster_header`'s. Combining the parent and child markers may improve classification results. 

In [ ]:
cluster_header = "cell_type"
subset_col = "feature_type"
subset_gene = "lncRNA"

#### Checking cell annotation sizes 

**Note:** Some datasets are too large and need to be downsampled to be run through the pipeline. When downsampling, be sure to have all the granular cluster annotations represented. 

In [ ]:
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

#### (Optional) Downsampling

In [ ]:
if to_downsample:
    adata = ct.samples.downsample_adata(adata_raw, mode = "total", n_cells = to_downsample_num, by = cluster_header,
                                        random_state = seed, return_index = False)
else:
    adata = adata_raw

In [ ]:
pd.DataFrame(adata.obs[cluster_header].value_counts()).reset_index()

#### Subsetting based on gene type.

In [ ]:
# Keep only lncRNA genes
adata_subset = adata[:, adata.var[subset_col] == subset_gene].copy()

print(adata.shape)
print(adata_subset.shape)

# Save
filename = file.replace(".h5ad", f"_subset_{subset_gene}.h5ad")
print(f"Saving subset anndata object as...\n{filename}")
adata_subset.write_h5ad(filename)

### Preprocessing

#### Generating scanpy dendrograms

**Note:** Only run if there is no pre-defined dendrogram order. This step can still be run with no effects, but the runtime may increase. 

Dendrogram order is stored in `adata.uns["dendrogram_cluster"]["categories_ordered"]`. 

In [ ]:
# full adata
if not adata.obsm or "X_pca" not in adata.obsm:
    sc.pp.pca(adata, random_state=seed)

ns.pp.dendrogram(adata, cluster_header, save = True, output_folder = output_folder, outputfilename_suffix = cluster_header)

In [ ]:
# subset adata
if not adata_subset.obsm or "X_pca" not in adata_subset.obsm:
    sc.pp.pca(adata_subset, random_state=seed)

ns.pp.dendrogram(adata_subset, cluster_header, save = True, output_folder = output_folder, outputfilename_suffix = cluster_header)

#### Calculating cluster medians per gene

Run `ns.pp.prep_medians` before running NS-Forest.

**Note:** Do **not** run if evaluating marker lists. Do **not** run when generating scanpy plots (e.g. dot plot, violin plot, matrix plot). 

In [ ]:
# full adata
adata = ns.pp.prep_medians(adata, cluster_header)
adata.varm[f"medians_{cluster_header}"].head()

In [ ]:
# subset adata
adata_subset = ns.pp.prep_medians(adata_subset, cluster_header)
adata_subset.varm[f"medians_{cluster_header}"].head()

#### Calculating binary scores per gene per cluster

Run `ns.pp.prep_binary_scores` before running NS-Forest. Do not need to run if evaluating marker lists. Do not need to run when generating scanpy plots. 

In [ ]:
# full adata
adata = ns.pp.prep_binary_scores(adata, cluster_header)
adata.varm[f"binary_scores_{cluster_header}"].head()

In [ ]:
# subset adata
adata_subset = ns.pp.prep_binary_scores(adata_subset, cluster_header)
adata_subset.varm[f"binary_scores_{cluster_header}"].head()

#### Saving preprocessed AnnData as new h5ad

In [ ]:
# full adata
filename = file.replace(".h5ad", "_preprocessed.h5ad")
print(f"Saving new anndata object as...\n{filename}")
adata.write_h5ad(filename)
adata

In [ ]:
# subset adata
filename = file.replace(".h5ad", f"_subset_{subset_gene}_preprocessed.h5ad")
print(f"Saving new anndata object as...\n{filename}")
adata.write_h5ad(filename)
adata

### Running NS-Forest

**Note:** Do not run NS-Forest if only evaluating input marker lists. 

In [ ]:
# full adata
outputfilename_prefix = cluster_header
results = ns.nsforesting.NSForest(adata, cluster_header, save_supplementary = True, save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)
results

In [ ]:
# subset adata
outputfilename_prefix_subset = cluster_header + "_subset_" + subset_gene
results_subset = ns.nsforesting.NSForest(adata_subset, cluster_header, save_supplementary = True, save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix_subset)
results_subset

### Visualization (Preprocessing)

In [ ]:
ns.pp.plot_varm(adata, f"medians_{cluster_header}", nonzero = True, save = True, output_folder = output_folder)
ns.pp.plot_varm(adata_subset, f"medians_{cluster_header}", nonzero = True, save = True, output_folder = output_folder)

In [ ]:
ns.pp.plot_varm(adata, f"medians_{cluster_header}", scale = "log", save = True, output_folder = output_folder)
ns.pp.plot_varm(adata_subset, f"medians_{cluster_header}", scale = "log", save = True, output_folder = output_folder)

In [ ]:
ns.pp.plot_varm(adata, f"binary_scores_{cluster_header}", nonzero = True, save = True, output_folder = output_folder)
ns.pp.plot_varm(adata_subset, f"binary_scores_{cluster_header}", nonzero = True, save = True, output_folder = output_folder)

In [ ]:
ns.pp.plot_varm(adata, f"binary_scores_{cluster_header}", scale = "log", save = True, output_folder = output_folder)
ns.pp.plot_varm(adata_subset, f"binary_scores_{cluster_header}", scale = "log", save = True, output_folder = output_folder)

### Visualization (Results): Plotting scanpy dot plot, violin plot, matrix plot for NS-Forest markers

**Note:** Assign pre-defined dendrogram order here **or** use `adata.uns["dendrogram_" + cluster_header]["categories_ordered"]`. 

In [ ]:
to_plot = results.copy()
to_plot_subset = results_subset.copy()

In [ ]:
# full adata
dendrogram = [] # custom dendrogram order
dendrogram = list(adata.uns["dendrogram_" + cluster_header]["categories_ordered"])
to_plot["clusterName"] = to_plot["clusterName"].astype("category")
to_plot["clusterName"] = to_plot["clusterName"].cat.set_categories(dendrogram)
to_plot = to_plot.sort_values("clusterName")
to_plot = to_plot.rename(columns = {"NSForest_markers": "markers"})

In [ ]:
# subset adata
dendrogram_subset = [] # custom dendrogram order
dendrogram_subset = list(adata.uns["dendrogram_" + cluster_header]["categories_ordered"])
to_plot_subset["clusterName"] = to_plot_subset["clusterName"].astype("category")
to_plot_subset["clusterName"] = to_plot_subset["clusterName"].cat.set_categories(dendrogram_subset)
to_plot_subset = to_plot_subset.sort_values("clusterName")
to_plot_subset = to_plot_subset.rename(columns = {"NSForest_markers": "markers"})

In [ ]:
# full adata
markers_dict = dict(zip(to_plot["clusterName"], to_plot["markers"]))
markers_dict

In [ ]:
# subset adata
markers_dict_subset = dict(zip(to_plot_subset["clusterName"], to_plot_subset["markers"]))
markers_dict_subset

In [ ]:
ns.pl.dotplot(adata, markers_dict, cluster_header, dendrogram = dendrogram, save = True, output_folder = output_folder, outputfilename_suffix = outputfilename_prefix)
ns.pl.dotplot(adata_subset, markers_dict_subset, cluster_header, dendrogram = dendrogram_subset, save = True, output_folder = output_folder, outputfilename_suffix = outputfilename_prefix_subset)

In [ ]:
ns.pl.stackedviolin(adata, markers_dict, cluster_header, dendrogram = dendrogram, save = True, output_folder = output_folder, outputfilename_suffix = outputfilename_prefix)
ns.pl.stackedviolin(adata_subset, markers_dict_subset, cluster_header, dendrogram = dendrogram_subset, save = True, output_folder = output_folder, outputfilename_suffix = outputfilename_prefix_subset)

In [ ]:
ns.pl.matrixplot(adata, markers_dict, cluster_header, dendrogram = dendrogram, save = True, output_folder = output_folder, outputfilename_suffix = outputfilename_prefix)
ns.pl.matrixplot(adata_subset, markers_dict_subset, cluster_header, dendrogram = dendrogram_subset, save = True, output_folder = output_folder, outputfilename_suffix = outputfilename_prefix_subset)

#### Plotting classification metrics from NS-Forest results

In [ ]:
ns.pl.boxplot(results, ["f_score", "precision", "recall", "onTarget"], save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)
ns.pl.boxplot(results_subset, ["f_score", "precision", "recall", "onTarget"], save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix_subset)

#### Plotting individual classification metrics

In [ ]:
ns.pl.boxplot(results, "f_score", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)
ns.pl.boxplot(results_subset, "f_score", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix_subset)

#### Plotting metrics vs clusterSize

In [ ]:
ns.pl.scatter_w_clusterSize(results, "f_score", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)
ns.pl.scatter_w_clusterSize(results_subset, "f_score", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix_subset)

In [ ]:
ns.pl.scatter_w_clusterSize(results, "precision", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)
ns.pl.scatter_w_clusterSize(results_subset, "precision", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix_subset)

In [ ]:
ns.pl.scatter_w_clusterSize(results, "recall", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)
ns.pl.scatter_w_clusterSize(results_subset, "recall", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix_subset)

In [ ]:
ns.pl.scatter_w_clusterSize(results, "onTarget", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix)
ns.pl.scatter_w_clusterSize(results_subset, "onTarget", save = True, output_folder = output_folder, outputfilename_prefix = outputfilename_prefix_subset)